# Approximate Multi-Qubit Toffoli with $\log(1/\epsilon)$ T Gates

This note presents an approximate $n$-qubit AND/Toffoli gate construction introduced in [GKZ25]. The idea is that, instead of directly computing the full AND, we randomly sample a small number of subsets of $[n]$, compute their parities, and combine those parity values through a smaller exact AND together with Clifford processing.

The idea is easiest to see in the simpler classical setting of the $n$-bit OR function. For each subset $S \subseteq [n]$, define
$$
\mathrm{XOR}_S(x)=\bigoplus_{i\in S} x_i
$$
to be the parity function. If $\mathrm{OR}_n(x)=0$, then every sampled parity is $0$. If $\mathrm{OR}_n(x)=1$, then for a uniformly random subset $S$, the parity $\mathrm{XOR}_S(x)$ equals $1$ with probability $1/2$. This leads to the following approximation scheme: If we sample independent subsets $S_1,\dots,S_k$ and compute
$$
g_{S_1,\dots,S_k}(x)=\mathrm{OR}_k\bigl(\mathrm{XOR}_{S_1}(x),\dots,\mathrm{XOR}_{S_k}(x)\bigr),
$$
then $g_{S_1,\dots,S_k}(x)$ always agrees with $\mathrm{OR}_n(x)$ when $\mathrm{OR}_n(x)=0$, and when $\mathrm{OR}_n(x)=1$ it fails only if all $k$ sampled parities are $0$, which happens with probability
$$
\Pr[g_{S_1,\dots,S_k}(x)\neq \mathrm{OR}_n(x)]\le 2^{-k}.
$$
Hence, $\mathrm{OR}_n$ can be approximated by computing a small number of random parities and then applying an exact OR on only $k$ bits. Performing this procedure in quantum superposition together with Clifford post-processing gives the approximate $n$-qubit Toffoli construction.

This note is organized as follows. First, we implement the vanilla log-depth exact construction from a balanced tree of 2-bit AND gates in `MultiAndLogDepth`, while keeping the same external interface as the bloq `MultiAnd`. Then, we move to implement the approximate construction in [GKZ25]. In the `ParityMask` bloq, we prepare the masked strings whose parities encode the sampled subsets. Then, in `ApproxMultiToffoli` we package the full approximate construction into a single bloq.

## `MultiAndLogDepth`

A log-depth $n$-bit AND with the same public I/O as `MultiAnd`.

This bloq computes the conjunction of an `n`-bit control register into a right-sided
output qubit using a balanced tree of Toffoli gates. The control register is
preserved, the output is exposed as `target`, and the intermediate tree values are
exposed as the right-sided `junk` register.

#### Parameters
 - `cvs`: A tuple of control values. Each entry must be `0` or `1`. The number of
   controls is `len(cvs)`, and must be at least 3.

#### Registers
 - `ctrl`: An `n`-bit control register.
 - `junk [right]`: An `n - 2` qubit junk register storing balanced-tree intermediate values.
 - `target [right]`: The output bit.

In [ ]:
from qualtran.bloqs.mcmt import MultiAndLogDepth

### Example Instances

In [ ]:
multi_and_log_depth = MultiAndLogDepth(cvs=(1, 1, 1, 1))

#### Graphical Signature

In [ ]:
from qualtran.drawing import show_bloqs

show_bloqs([multi_and_log_depth], ['`multi_and_log_depth`'])

### Decomposition

In [ ]:
from qualtran.drawing import show_bloq

show_bloq(multi_and_log_depth.decompose_bloq())

### Call Graph

In [ ]:
from qualtran.drawing import show_call_graph, show_counts_sigma
from qualtran.resource_counting.generalizers import ignore_split_join

g, sigma = multi_and_log_depth.call_graph(max_depth=1, generalizer=ignore_split_join)
show_call_graph(g)
show_counts_sigma(sigma)

## `ParityMask`

Prepare masked copies of an input register for sampled subset-parity computation.

This bloq takes an `(n - 1)`-qubit input register `x` and prepares `k` right-output
registers `x_0, ..., x_{k-1}`. Each output register is selected by one classical
mask from `sample_strings`.

For mask `sample_strings[i]`, the output bit is

$$
[x_i]_j = x_j \cdot s[i]_j.
$$

Thus, the parity of `x_i` is the parity of the subset of input bits selected by
the `i`-th sampled mask. This is a sample-preparation subroutine for the
approximate multi-controlled AND / Toffoli construction, where sampled subset
parities are later combined by a smaller exact AND.

#### Parameters

 - `n`: The total number of qubits in the target approximate Toffoli construction. This bloq acts on registers of length `n - 1`.
 - `k`: The number of sampled subsets, equivalently the number of output registers.
 - `sample_strings`: A tuple of `k` classical bit strings, each of length `n - 1`. The entry `sample_strings[i][j]` is `1` iff the `j`-th input bit is included in the `i`-th sampled subset.

#### Registers

 - `x`: An `(n - 1)`-qubit input register.
 - `x_i [right]`: For each `i = 0, ..., k - 1`, an `(n - 1)`-qubit output register containing the masked copy selected by `sample_strings[i]`.

In [ ]:
from qualtran.bloqs.mcmt import ParityMask

### Example Instances

In [ ]:
sample_strings = (
    (1, 0, 1, 0),
    (0, 1, 1, 0),
    (1, 1, 0, 1),
)

parity_mask = ParityMask(n=5, k=3, sample_strings=sample_strings)

#### Graphical Signature

In [ ]:
from qualtran.drawing import show_bloqs

show_bloqs([parity_mask], ['`parity_mask`'])

### Decomposition

In [ ]:
from qualtran.drawing import show_bloq

show_bloq(parity_mask.decompose_bloq())

### Call Graph

In [ ]:
from qualtran.drawing import show_call_graph, show_counts_sigma

parity_mask_g, parity_mask_sigma = parity_mask.call_graph(max_depth=1)
show_call_graph(parity_mask_g)
show_counts_sigma(parity_mask_sigma)

## `ApproxMultiToffoli`

Approximate multi-controlled Toffoli via sampled subset parities.

This bloq implements the approximate construction by first using `ParityMask` to prepare masked copies of the control register. It then computes the parity of each masked copy, negates those parity bits, and applies a smaller exact AND to the resulting `k` bits.

The output approximates the `(n - 1)`-bit AND of the control register, or equivalently an `n`-qubit Toffoli with `(n - 1)` controls and one target. In this implementation, the target is exposed as a fresh right-sided output qubit rather than toggling an input target in place.

#### Parameters

 - `n`: The total number of qubits in the target Toffoli interpretation. The bloq acts on an `(n - 1)`-qubit control register.
 - `k`: The number of sampled subsets used in the approximation.
 - `sample_strings`: A tuple of `k` classical bit strings, each of length `n - 1`. The entry `sample_strings[i][j]` is `1` iff the `j`-th control bit is included in the `i`-th sampled subset.

#### Registers

 - `ctrl`: An `(n - 1)`-qubit control register.
 - `junk [right]`: A right-sided junk register containing the masked copies, parity values, and auxiliary qubits used by the exact AND subroutine.
 - `target [right]`: The output bit of the approximate multi-controlled AND.

In [ ]:
from qualtran.bloqs.mcmt import ApproxMultiToffoli

### Example Instances

In [ ]:
n = 3
k = 2
sample_strings = (
    (1, 0),
    (0, 1),
)

approx_multi_toffoli = ApproxMultiToffoli(n=n, k=k, sample_strings=sample_strings)

#### Graphical Signature

In [ ]:
from qualtran.drawing import show_bloqs

show_bloqs([approx_multi_toffoli], ['`approx_multi_toffoli`'])

### Decomposition

In [ ]:
from qualtran.drawing import show_bloq

show_bloq(approx_multi_toffoli.decompose_bloq())

### Call Graph

In [ ]:
from qualtran.drawing import show_call_graph, show_counts_sigma

approx_multi_toffoli_g, approx_multi_toffoli_sigma = approx_multi_toffoli.call_graph(max_depth=1)
show_call_graph(approx_multi_toffoli_g)
show_counts_sigma(approx_multi_toffoli_sigma)

Finally, we test the correctness of the algorithm on classical inputs/outputs. In the example below, we set $n=17$ and $k=4$. We randomly sample $x$ and sample_strings every time. The expected error probability is $2^{-k}=1/16$.

In [ ]:
import random

# n = 17 (16 controls), k = 4
n = 17
k = 4
m = n - 1
ALL_ONES = (1 << m) - 1

def random_sample_string(width: int) -> tuple[int, ...]:
    return tuple(random.randint(0, 1) for _ in range(width))

def int_to_bits(x: int, width: int) -> tuple[int, ...]:
    return tuple((x >> j) & 1 for j in range(width))

def ideal_toffoli_t(x_int: int, t_in: int) -> int:
    # New ApproxMultiToffoli outputs a fresh target bit
    return int(x_int == ALL_ONES)

# ----------------------------
# Single randomized example
# ----------------------------

# Randomly sample x
x_int = random.randint(0, ALL_ONES)
t_in = 0

# Fresh random sample strings
sample_strings = tuple(random_sample_string(m) for _ in range(k))

B = ApproxMultiToffoli(n=n, k=k, sample_strings=sample_strings)

out = B.call_classically(
    ctrl=int_to_bits(x_int, m),
)

t_out = out[2]
t_expected = ideal_toffoli_t(x_int, t_in)

print("Single randomized test case")
print(f"  x (binary) = {x_int:016b}")
print(f"  sample_strings   = {sample_strings}")
print(f"  Approx t  = {t_out}")
print(f"  Ideal t   = {t_expected}")
print(f"  MATCH     = {t_out == t_expected}")